In [17]:
import pandas as pd
from pathlib import Path
import sys

sys.path.insert(1, r"..\config")

In [18]:
from paths import DISC_STANDARDS_RAW

raw = pd.read_excel(DISC_STANDARDS_RAW / "EFRAG IG 3 List of ESRS Data Points.xlsx", skiprows=1, sheet_name='ESRS E1')

raw.head()

,ID,ESRS,DR,Paragraph,Related AR,Name,Data Type,Conditional or alternative DP,May \n[V],Appendix B - ESRS 2 \n(SFDR + PILLAR 3 + Benchmark + CL),Appendix C - ESRS 1\nDPs subject to phasing-in provisions applicable to undertaking with less than 750 employees,Appendix C - ESRS 1\nDPs subject to phasing-in provisions applicable to all undertakings
0,E1.GOV-3_01,E1,E1.GOV-3,13,NaN,Disclosure of whether and how climate-related ...,narrative,NaN,NaN,NaN,NaN,NaN
1,E1.GOV-3_02,E1,E1.GOV-3,13,NaN,Percentage of remuneration recognised that is ...,percent,NaN,NaN,NaN,NaN,NaN
2,E1.GOV-3_03,E1,E1.GOV-3,13,NaN,Explanation of climate-related considerations ...,narrative,NaN,NaN,NaN,NaN,NaN
3,E1-1_01,E1,E1-1,14,AR 1,Disclosure of transition plan for climate cha...,narrative,NaN,NaN,CL,NaN,NaN
4,E1-1_02,E1,E1-1,16 a,AR 2,Explanation of how targets are compatible with...,narrative,NaN,NaN,NaN,NaN,NaN


In [19]:
req_keys = [
    'ESRS',
    'DR',
    'Paragraph',
    'Related AR',
    'Name',
    'Data Type',
    'Conditional or alternative DP'
]

unfiltered_dict = raw.loc[:, req_keys]

unfiltered_dict = unfiltered_dict.astype(str)

In [20]:
unfiltered_dict = unfiltered_dict.apply(lambda x: x.str.strip())

In [21]:
unfiltered_dict.head()

,ESRS,DR,Paragraph,Related AR,Name,Data Type,Conditional or alternative DP
0,E1,E1.GOV-3,13,nan,Disclosure of whether and how climate-related ...,narrative,nan
1,E1,E1.GOV-3,13,nan,Percentage of remuneration recognised that is ...,percent,nan
2,E1,E1.GOV-3,13,nan,Explanation of climate-related considerations ...,narrative,nan
3,E1,E1-1,14,AR 1,Disclosure of transition plan for climate cha...,narrative,nan
4,E1,E1-1,16 a,AR 2,Explanation of how targets are compatible with...,narrative,nan


In [22]:
nested = (
    unfiltered_dict.groupby(["ESRS", "DR", "Paragraph"], dropna=False)
      .apply(lambda g: g[["Related AR", "Name", "Data Type", "Conditional or alternative DP"]]
             .rename(columns={
                 "Related AR": "AR",
                 "Data Type": "datatype",
                 "Conditional or alternative DP": "dp_type"
                 })
             .to_dict(orient="records"), include_groups=False)
      .reset_index(name="entries")
)

In [23]:
level1 = (
    nested.groupby("ESRS")
    .apply(lambda g: g.groupby("DR")
           .apply(lambda h: dict(zip(h["Paragraph"], h["entries"])), include_groups=False)
           .to_dict(), include_groups=False)
    .to_dict()
)

In [24]:
level1

{'E1': {'E1-1': {'14': [{'AR': 'AR 1',
     'Name': 'Disclosure of transition plan  for climate change mitigation',
     'datatype': 'narrative',
     'dp_type': 'nan'}],
   '16 a': [{'AR': 'AR 2',
     'Name': 'Explanation of how targets are compatible with limiting of global warming to one and half degrees Celsius in line with Paris Agreement',
     'datatype': 'narrative',
     'dp_type': 'nan'}],
   '16 b': [{'AR': 'nan',
     'Name': 'Disclosure of decarbonisation levers and key action',
     'datatype': 'narrative',
     'dp_type': 'nan'}],
   '16 c': [{'AR': 'nan',
     'Name': 'Disclosure of significant operational expenditures (Opex) and (or) capital expenditures (Capex) required for implementation of action plan',
     'datatype': 'narrative',
     'dp_type': 'nan'},
    {'AR': 'nan',
     'Name': 'Financial resources allocated to action plan (OpEx)',
     'datatype': 'monetary',
     'dp_type': 'nan'},
    {'AR': 'nan',
     'Name': 'Financial resources allocated to action p